In [1]:
import pyspark
from pyspark.sql import SparkSession

### Answer 1

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()
print(f"Spark version is {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/06 22:40:45 WARN Utils: Your hostname, cedan-VMware20-1, resolves to a loopback address: 127.0.1.1; using 192.168.99.132 instead (on interface ens160)
26/03/06 22:40:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/06 22:40:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version is 4.1.1


In [5]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-06 22:10:16--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.160.52.39, 3.160.52.113, 3.160.52.101, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.160.52.39|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  6.13MB/s    in 13s     

2026-03-06 22:10:30 (5.10 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



### Answer 2

In [3]:
df= spark.read \
    .option("header", "true")\
    .parquet('yellow_tripdata_2025-11.parquet')

In [11]:
df=df.repartition(4)

In [12]:
df.write.parquet('data/pq/yellow/2025/11',mode='overwrite')

In [14]:
from pathlib import Path

files = [f.stat().st_size for f in Path('data/pq/yellow/2025/11').glob('*.parquet')]

if files:
    avg_mb = (sum(files) / len(files)) / (1024**2)
    print(f"Total Parquet Files: {len(files)}")
    print(f"Average Size: {avg_mb:.2f} MB")
else:
    print("No .parquet files found.")

Total Parquet Files: 4
Average Size: 24.41 MB


### Answer 3

In [4]:
df_yellow=spark.read.parquet('data/pq/yellow/2025/11')

In [5]:
df_yellow.createOrReplaceTempView('yellow')

In [6]:
df_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [7]:
df_yellow_trip_count=spark.sql(
    """ 
    SELECT 
        COUNT (*) AS trip_count
    FROM
        yellow
    WHERE
        tpep_pickup_datetime 
    BETWEEN
        '2025-11-15 00:00:00' AND '2025-11-15 23:59:59'
    """)\
    .show()

+----------+
|trip_count|
+----------+
|    162604|
+----------+



### Answer 4

In [11]:
df_yellow_longest_trip=spark.sql(
    """ 
    SELECT 
        MAX((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600) AS longest_trip
    FROM yellow
    """)\
    .show()

+-----------------+
|     longest_trip|
+-----------------+
|90.64666666666666|
+-----------------+



### Answer 5

Spark's UI works on 4040 port

### Answer 6

In [12]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-06 22:56:49--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.155.128.46, 18.155.128.187, 18.155.128.6, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.155.128.46|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-06 22:56:49 (428 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [14]:
df_zones=spark.read \
    .option("header", "true")\
    .csv('taxi_zone_lookup.csv')

In [22]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [23]:
df_zones.createOrReplaceTempView('zones')

In [25]:
df_least_frequent_zone=spark.sql(
    """ 
    SELECT 
        zones.ZONE AS zone, 
        COUNT (*) trip_count
    FROM 
        yellow
    JOIN 
        zones ON yellow.PULocationID = zones.LocationID
    GROUP BY 
        zone
    ORDER BY
        trip_count ASC
    LIMIT 1
    
    """).show()

+-------------+----------+
|         zone|trip_count|
+-------------+----------+
|Arden Heights|         1|
+-------------+----------+

